# 03 — IFRS 9 stage movements

**Plain English:** Banks report loans in three IFRS 9 stages and care a lot about
*movement* between them — every 1→2 is a new "significant increase in credit
risk" flag, every 2→3 is a new default, every 2→1 is a cure. Here we count those
month-over-month moves and show the mix.

**One-line terms**
- **Stage 1 → 2** — loan deteriorated into the watch list (SICR).
- **Stage 2 → 3** — loan defaulted.
- **Stage 2 → 1 / 3 → 2** — loan cured / partially cured.

This is the same engine as notebook 02, read through the IFRS 9 lens regulators use.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# Period-over-period stage classification (where loan-months sit) --------
stage_mix = (panel[panel.stage.notna()]
             .groupby(["vintage", "stage"]).size()
             .unstack(fill_value=0))
stage_mix = stage_mix.div(stage_mix.sum(1), axis=0).round(4)
stage_mix

stage,Stage 1,Stage 2,Stage 3
vintage,,,
2007,0.9030,0.0427,0.0543
2008,0.9347,0.0310,0.0343
2015,0.9851,0.0086,0.0062


In [3]:
# Stage-movement summary (the headline counts) --------------------------
moves = m.stage_movement_summary(panel)
moves["share"] = moves["share"].round(4)
moves

,move,loan_months,share
0,1 -> 1 stay performing,8124369,0.9198
1,3 -> 3 stay defaulted,250832,0.0284
2,2 -> 2 stay watch,145608,0.0165
3,1 -> exit (prepaid),124177,0.0141
4,1 -> 2 deteriorate (SICR),83246,0.0094
5,2 -> 1 cure,63194,0.0072
6,2 -> 3 deteriorate (default),23073,0.0026
7,3 -> 1 cure,10873,0.0012
8,3 -> 2 partial cure,4645,0.0005
9,2 -> exit (prepaid),899,0.0001


In [4]:
# Save the clean table ---------------------------------------------------
moves.to_csv(m.OUT_TABLES / "03_stage_movements.csv", index=False)
stage_mix.to_csv(m.OUT_TABLES / "03_stage_mix_by_vintage.csv")
print("saved stage movements + stage mix")

saved stage movements + stage mix
